## Semi-Supervised Learning

Most times, manually labeling a complete dataset can be very time consuming, and in
real life scenarios, time is money. 

We are aiming to partially delete the target feature in
different proportions for each class, and run active and pasive semi-supervised learning
processes. This experiment will help us understand how it performs with our samples.


Our set of samples is not as large as we would like it to be, but it will be useful to study
how to identify the most interesting samples to manually label and leave the rest to the
model, saving the time and effort it takes to fully supervise a dataset

In [18]:
import pandas
import numpy as np


In [19]:
data = pandas.read_csv("../data/updated_pollution_dataset.csv")

In [20]:
target_col = data.columns[-1]

distribution = (
    data[target_col]
    .value_counts(dropna=False)
    .rename_axis(target_col)
    .reset_index(name="count")
)

distribution["percentage"] = (distribution["count"] / len(data) * 100).round(2)
distribution

,Air Quality,count,percentage
0,Good,2000,40.0
1,Moderate,1500,30.0
2,Poor,1000,20.0
3,Hazardous,500,10.0


In [21]:
quality_map = {
    "Good": 3,
    "Moderate": 2,
    "Poor": 1,
    "Hazardous": 0
}

data[target_col] = data[target_col].map(quality_map)


data[target_col] = data[target_col].astype("int8")
data[[target_col]].head()


,Air Quality
0,2
1,2
2,2
3,3
4,3


* Target variable partial deletion

1. Random Deletion

In [22]:
label_ratio = 0.1 # 10% of the data will be labeled
RANDOM_STATE = 42

data["true_label"] = data[target_col] # true labels for evaluation



PASSIVE Semi-Supervision

We ignore the class distribution and pseudo-randomly mask target samples

In [23]:
def passive_mask(data, target, label_ratio=0.1, random_state=42):
    data = data.copy(deep=True)
    unlabeled_idx = data.sample(frac=1 - label_ratio, random_state=random_state).index
    data.loc[unlabeled_idx, target] = np.nan
    return data

In [24]:
def active_mask(data, target, label_ratio, random_state=42):
    
    data = data.copy(deep=True)

    unlabeled_idx = (
        data.groupby(target, group_keys=False)          # group by class
          .apply(lambda g: g.sample(                  # from each class
              frac=1 - label_ratio,                 # mask this fraction
              random_state=random_state
          ))
          .index
    )
    data.loc[unlabeled_idx, target] = np.nan
    return data

In [25]:
data_passive = passive_mask(data, target_col, label_ratio=label_ratio, random_state=RANDOM_STATE)
data_active  = active_mask(data,  target_col, label_ratio=label_ratio, random_state=RANDOM_STATE)

In [26]:
for name, data in [("Passive", data_passive), ("Active", data_active)]:
    labeled   = data[target_col].notna()
    print(f"\n{'─'*40}")
    print(f"{name} - Labeled: {labeled.sum()} | Unlabeled: {(~labeled).sum()}")
    print("Label distribution (labeled only):")
    print(data.loc[labeled, target_col].value_counts(normalize=True).sort_index())


────────────────────────────────────────
Passive - Labeled: 500 | Unlabeled: 4500
Label distribution (labeled only):
Air Quality
0.0    0.106
1.0    0.198
2.0    0.306
3.0    0.390
Name: proportion, dtype: float64

────────────────────────────────────────
Active - Labeled: 500 | Unlabeled: 4500
Label distribution (labeled only):
Air Quality
0.0    0.1
1.0    0.2
2.0    0.3
3.0    0.4
Name: proportion, dtype: float64


Quality map for help:

quality_map = {
    "Good": 3,
    "Moderate": 2,
    "Poor": 1,
    "Hazardous": 0
}

In [27]:
data_active.head()


,Temperature,Humidity,PM2.5,PM10,NO2,SO2,CO,Proximity_to_Industrial_Areas,Population_Density,Air Quality,true_label
0,29.8,59.1,5.2,17.9,18.9,9.2,1.72,6.3,319,NaN,2
1,28.3,75.6,2.3,12.2,30.8,9.7,1.64,6.0,611,NaN,2
2,23.1,74.7,26.7,33.8,24.4,12.6,1.63,5.2,619,NaN,2
3,27.1,39.1,6.1,6.3,13.5,5.3,1.15,11.1,551,NaN,3
4,26.5,70.7,6.9,16.0,21.9,5.6,1.01,12.7,303,NaN,3


In [28]:
data_passive.head()

,Temperature,Humidity,PM2.5,PM10,NO2,SO2,CO,Proximity_to_Industrial_Areas,Population_Density,Air Quality,true_label
0,29.8,59.1,5.2,17.9,18.9,9.2,1.72,6.3,319,NaN,2
1,28.3,75.6,2.3,12.2,30.8,9.7,1.64,6.0,611,NaN,2
2,23.1,74.7,26.7,33.8,24.4,12.6,1.63,5.2,619,NaN,2
3,27.1,39.1,6.1,6.3,13.5,5.3,1.15,11.1,551,NaN,3
4,26.5,70.7,6.9,16.0,21.9,5.6,1.01,12.7,303,3.0,3


data split

In [29]:
# Labeled pool - train supervised component
labeled_data = data_active[data_active[target_col].notna()]

# Unlabeled pool - semi-supervised propagation / pseudo-labeling
unlabeled_data = data_active[data_active[target_col].isna()]

# Ground truth  → final evaluation (never touch during training!)
y_true_unlabeled = data_active.loc[unlabeled_data.index, "true_label"]

Feature Preparation

In [30]:
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in data.columns if c not in [target_col, "true_label"]]

scaler = StandardScaler()

def prepare_xy(df):
    labeled   = df[df[target_col].notna()]
    unlabeled = df[df[target_col].isna()]
    X_lab   = scaler.fit_transform(labeled[feature_cols])
    y_lab   = labeled[target_col].astype(int).values
    X_unlab = scaler.transform(unlabeled[feature_cols])
    y_true  = df.loc[unlabeled.index, "true_label"].astype(int).values
    return X_lab, y_lab, X_unlab, y_true

X_lab, y_lab, X_unlab, y_true = prepare_xy(data_active)

Supervised baseline

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
clf_base.fit(X_lab, y_lab)

y_pred_base = clf_base.predict(X_unlab)
print("=== Supervised Baseline (labeled only) ===")
print(classification_report(y_true, y_pred_base,
      target_names=["3","2","1","0"]))

=== Supervised Baseline (labeled only) ===
              precision    recall  f1-score   support

           3       0.83      0.83      0.83       450
           2       0.86      0.83      0.84       900
           1       0.94      0.96      0.95      1350
           0       1.00      1.00      1.00      1800

    accuracy                           0.94      4500
   macro avg       0.91      0.90      0.91      4500
weighted avg       0.94      0.94      0.94      4500



Semi-supervised learning

Label Spreading (passive vs active)

In [36]:
from sklearn.semi_supervised import LabelSpreading

results = {}

for name, df in [("Passive", data_passive), ("Active", data_active)]:
    X_lab_i, y_lab_i, X_unlab_i, y_true_i = prepare_xy(df)

    # Build full X and y (-1 marks unlabeled)
    X_all = np.vstack([X_lab_i, X_unlab_i])
    y_all = np.concatenate([y_lab_i, np.full(len(X_unlab_i), -1)])

    ls = LabelSpreading(kernel="knn", n_neighbors=7, max_iter=100)
    ls.fit(X_all, y_all)

    n_unlab = len(X_unlab_i)
    y_pred  = ls.transduction_[-n_unlab:]
    results[f"LabelSpreading_{name}"] = (y_true_i, y_pred)
    print(f"\n=== Label Spreading — {name} ===")
    print(classification_report(y_true_i, y_pred,
          target_names=["3","2","1","0"]))


=== Label Spreading — Passive ===
              precision    recall  f1-score   support

           3       0.79      0.60      0.68       447
           2       0.72      0.72      0.72       901
           1       0.87      0.91      0.89      1347
           0       0.98      1.00      0.99      1805

    accuracy                           0.88      4500
   macro avg       0.84      0.81      0.82      4500
weighted avg       0.87      0.88      0.87      4500


=== Label Spreading — Active ===
              precision    recall  f1-score   support

           3       0.78      0.67      0.72       450
           2       0.75      0.74      0.74       900
           1       0.89      0.92      0.90      1350
           0       0.98      1.00      0.99      1800

    accuracy                           0.89      4500
   macro avg       0.85      0.83      0.84      4500
weighted avg       0.89      0.89      0.89      4500



Pseudo Labels Self Training

In [41]:
from sklearn.semi_supervised import SelfTrainingClassifier

for name, df in [("Passive", data_passive), ("Active", data_active)]:
    X_lab_i, y_lab_i, X_unlab_i, y_true_i = prepare_xy(df)

    X_all = np.vstack([X_lab_i, X_unlab_i])
    y_all = np.concatenate([y_lab_i, np.full(len(X_unlab_i), -1)])

    base_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    st = SelfTrainingClassifier(base_clf, threshold=0.8, verbose=False)
    st.fit(X_all, y_all)

    n_unlab = len(X_unlab_i)
    y_pred  = st.transduction_[-n_unlab:]
    labeled_mask = y_pred != -1
    n_unlabeled_remaining = (~labeled_mask).sum()
    if n_unlabeled_remaining > 0:
        print(f"{n_unlabeled_remaining} samples could not be pseudo-labeled (kept as -1)")

    y_pred_clean  = y_pred[labeled_mask]
    y_true_clean  = y_true_i[labeled_mask]

    results[f"SelfTraining_{name}"] = (y_true_clean, y_pred_clean)

    print(f"\n=== Self-Training — {name} ===")
    print(classification_report(
        y_true_clean, y_pred_clean,
        labels=[0, 1, 2, 3],
        target_names=["3","2","1","0"]
    ))

376 samples could not be pseudo-labeled (kept as -1)

=== Self-Training — Passive ===
              precision    recall  f1-score   support

           3       0.88      0.91      0.89       346
           2       0.91      0.86      0.89       698
           1       0.96      0.98      0.97      1280
           0       1.00      1.00      1.00      1800

    accuracy                           0.96      4124
   macro avg       0.94      0.94      0.94      4124
weighted avg       0.96      0.96      0.96      4124

336 samples could not be pseudo-labeled (kept as -1)

=== Self-Training — Active ===
              precision    recall  f1-score   support

           3       0.90      0.92      0.91       389
           2       0.88      0.85      0.87       677
           1       0.95      0.96      0.96      1309
           0       1.00      1.00      1.00      1789

    accuracy                           0.96      4164
   macro avg       0.93      0.93      0.93      4164
weighted avg  

In [ ]:
#TODO Corregir el tema del pseudo label, originalmente los maskeaba con el valor NaN, pero se vuelve loco con -1s

Comparison

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

rows = [("Baseline (supervised)", y_true, y_pred_base)]
for k, (yt, yp) in results.items():
    rows.append((k, yt, yp))

summary = pd.DataFrame([{
    "Model": name,
    "Accuracy": accuracy_score(yt, yp),
    "Macro F1": f1_score(yt, yp, average="macro"),
} for name, yt, yp in rows])

print(summary.to_string(index=False))